# Audit sumber data Siklus 2 ER CyRIS

**Jalankan di Google Colab akun yang memuat folder `MyDrive/ercyris_siklus2`.** Notebook ini memeriksa keterulangan sampel Paket A dan jumlah kandidat observasi yang belum masuk sampel 30.000. Notebook hanya membaca empat sumber dan menulis ringkasan JSON kecil pada subfolder baru `audit_koreksi_siklus2`; tidak melatih model, menilai gerbang baru, atau mengubah hasil lama. Membaca HDFS.log yang besar dapat memakan waktu.

**Penting:** baris yang berada di luar sampel lama belum otomatis memenuhi syarat holdout baru. Kecocokan identitas, duplikasi, periode, dan mutu observasi perlu diaudit setelah ringkasan ini diperoleh. Jangan menafsirkan `missing_fraction` sebagai `q_e`.


In [ ]:
import hashlib
import json
import re
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

from google.colab import drive
drive.mount('/content/drive')

# Notebook awal memakai path ini. Bila folder hanya berada di “Shared with me”,
# tambahkan shortcut folder ke My Drive atau ubah BASE ke lokasi yang tampil di Colab.
BASE = Path('/content/drive/MyDrive/ercyris_siklus2')
OUT_DIR = BASE / 'audit_koreksi_siklus2'
PATHS = {
    'BGL': BASE / 'BGL.log',
    'HDFS': BASE / 'HDFS.log',
    'HDFS_LABEL': BASE / 'anomaly_label.csv',
    'UNSW-NB15': BASE / 'UNSW-NB15_1.csv',
    'UNSW-NB15_2': BASE / 'UNSW-NB15_2.csv',
}
PRIMARY_DATASETS = ['BGL', 'HDFS', 'UNSW-NB15']
INCLUDE_CICIDS_CONTRAST = False
MAX_SAMPLES_PER_DATASET = 30_000
BGL_PARSE_LIMIT = 250_000

for key in ['BGL', 'HDFS', 'HDFS_LABEL', 'UNSW-NB15']:
    if not PATHS[key].is_file():
        raise FileNotFoundError(
            f'{PATHS[key]} tidak ditemukan. Periksa BASE; tidak perlu mengunggah log.'
        )
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('Sumber ditemukan:', [(key, PATHS[key].stat().st_size) for key in PATHS if PATHS[key].is_file()])
print('Folder ringkasan:', OUT_DIR)


## Utilitas dan loader yang sama dengan notebook benchmark sebelumnya

Dua sel berikut disalin apa adanya dari notebook `ER_CyRIS_Algorithmic_Diversity_Benchmark_v1.ipynb`. Tujuannya agar identifikasi 30.000 baris yang lama tidak memakai definisi parser yang baru.


In [ ]:
"""Core utilities for the ER-CyRIS algorithmic-diversity benchmark."""

from __future__ import annotations

import hashlib
import json

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split


def _as_binary_vector(y) -> np.ndarray:
    values = np.asarray(y, dtype=int).reshape(-1)
    if len(values) < 8:
        raise ValueError("At least eight observations are required")
    if set(np.unique(values)) != {0, 1}:
        raise ValueError("Binary labels must contain both 0 and 1")
    return values


def make_four_way_split(y, seed: int = 42) -> dict[str, np.ndarray]:
    """Return disjoint 55/15/15/15 stratified index partitions."""
    labels = _as_binary_vector(y)
    indices = np.arange(len(labels))
    train_idx, remainder_idx = train_test_split(
        indices,
        train_size=0.55,
        random_state=seed,
        stratify=labels,
    )
    middle_idx, holdout_idx = train_test_split(
        remainder_idx,
        test_size=1.0 / 3.0,
        random_state=seed + 1,
        stratify=labels[remainder_idx],
    )
    calibration_idx, development_idx = train_test_split(
        middle_idx,
        test_size=0.5,
        random_state=seed + 2,
        stratify=labels[middle_idx],
    )
    return {
        "train": np.sort(train_idx),
        "calibration": np.sort(calibration_idx),
        "development": np.sort(development_idx),
        "holdout": np.sort(holdout_idx),
    }


def add_gaussian_noise(X, sigma: float, seed: int) -> np.ndarray:
    values = np.asarray(X, dtype=float)
    if sigma < 0:
        raise ValueError("sigma must be non-negative")
    if sigma == 0:
        return values.copy()
    noise = np.random.default_rng(seed).normal(0.0, sigma, size=values.shape)
    return np.clip(values + noise, 0.0, 1.0)


def expected_calibration_error(y_true, probabilities, n_bins: int = 10) -> float:
    labels = np.asarray(y_true, dtype=int).reshape(-1)
    probs = np.clip(np.asarray(probabilities, dtype=float).reshape(-1), 0.0, 1.0)
    if len(labels) != len(probs) or len(labels) == 0:
        raise ValueError("Labels and probabilities must have the same non-zero length")
    if n_bins < 1:
        raise ValueError("n_bins must be positive")

    edges = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.minimum(np.digitize(probs, edges[1:-1], right=False), n_bins - 1)
    ece = 0.0
    for bin_id in range(n_bins):
        mask = bin_ids == bin_id
        if not np.any(mask):
            continue
        accuracy = float(labels[mask].mean())
        confidence = float(probs[mask].mean())
        ece += float(mask.mean()) * abs(accuracy - confidence)
    return float(ece)


def select_f1_threshold(y_true, probabilities) -> tuple[float, float]:
    labels = np.asarray(y_true, dtype=int).reshape(-1)
    probs = np.clip(np.asarray(probabilities, dtype=float).reshape(-1), 0.0, 1.0)
    if len(labels) != len(probs) or len(labels) == 0:
        raise ValueError("Labels and probabilities must have the same non-zero length")

    candidates = np.unique(probs)
    scored = [
        (float(f1_score(labels, probs >= threshold, zero_division=0)), float(threshold))
        for threshold in candidates
    ]
    best_score = max(score for score, _ in scored)
    tied = [threshold for score, threshold in scored if score == best_score]
    best_threshold = min(tied, key=lambda value: (abs(value - 0.5), value))
    return float(best_threshold), float(best_score)


class PlattCalibrator:
    """One-dimensional sigmoid calibration fitted only on the calibration split."""

    def __init__(self, seed: int = 42):
        self.seed = int(seed)
        self.model = LogisticRegression(random_state=self.seed, solver="lbfgs")
        self._fitted = False

    def fit(self, scores, y):
        score_array = np.asarray(scores, dtype=float).reshape(-1, 1)
        labels = np.asarray(y, dtype=int).reshape(-1)
        if len(score_array) != len(labels) or set(np.unique(labels)) != {0, 1}:
            raise ValueError("Calibration data must contain aligned binary classes")
        self.model.fit(score_array, labels)
        self._fitted = True
        return self

    def predict_proba(self, scores) -> np.ndarray:
        if not self._fitted:
            raise RuntimeError("Call fit before predict_proba")
        score_array = np.asarray(scores, dtype=float).reshape(-1, 1)
        return self.model.predict_proba(score_array)[:, 1]


def stratified_cap_indices(y, max_samples: int, seed: int) -> np.ndarray:
    labels = _as_binary_vector(y)
    if max_samples < 2:
        raise ValueError("max_samples must be at least two")
    if len(labels) <= max_samples:
        return np.arange(len(labels), dtype=int)
    chosen, _ = train_test_split(
        np.arange(len(labels)),
        train_size=int(max_samples),
        random_state=seed,
        stratify=labels,
    )
    return np.sort(chosen)


def expected_run_count(
    n_datasets: int,
    n_models: int,
    n_representations: int,
    n_seeds: int,
    n_conditions: int,
) -> dict[str, int]:
    model_fits = int(n_datasets * n_models * n_representations * n_seeds)
    return {
        "model_fits": model_fits,
        "evaluation_rows": int(model_fits * n_conditions),
    }


def build_run_id(
    dataset: str,
    representation: str,
    model: str,
    seed: int,
    condition: str,
    split: str,
) -> str:
    return (
        f"{dataset}__{representation}__{model}__seed-{int(seed)}"
        f"__{condition}__{split}"
    )


def freeze_registry_digest(records: list[dict]) -> str:
    canonical_records = sorted(
        (json.dumps(record, sort_keys=True, separators=(",", ":")) for record in records)
    )
    payload = "\n".join(canonical_records).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


In [ ]:
def stable_bucket(value, modulus=1000):
    digest = hashlib.blake2b(str(value).encode('utf-8'), digest_size=8).digest()
    return int.from_bytes(digest, 'big') % modulus


def load_bgl(path):
    severity = {
        'INFO': 1, 'DEBUG': 0, 'WARN': 2, 'WARNING': 2, 'ERROR': 3,
        'SEVERE': 4, 'FATAL': 4, 'APPFPO': 3, 'KERN': 2,
        'KERNSVC': 3, 'APPFPW': 2,
    }
    attack_words = [
        'fatal', 'error', 'fail', 'exception', 'abort', 'illegal',
        'segfault', 'panic', 'crash', 'out of memory', 'overflow',
    ]
    rows = []
    with open(path, 'r', errors='replace') as stream:
        for line_number, line in enumerate(stream):
            if BGL_PARSE_LIMIT and line_number >= BGL_PARSE_LIMIT:
                break
            parts = line.split(None, 8)
            if len(parts) < 9:
                continue
            label = 0 if parts[0] == '-' else 1
            sev = severity.get(parts[6].upper(), 1)
            message = parts[8].lower()
            try:
                hour = int(parts[3].split(':')[0]) if ':' in parts[3] else 12
            except Exception:
                hour = 12
            rows.append({
                'severity': sev,
                'is_error': int(sev >= 3),
                'is_warning': int(sev == 2),
                'hour': hour,
                'off_hour': int(hour < 6),
                'peak_hour': int(8 <= hour <= 18),
                'attack_keyword': int(any(word in message for word in attack_words)),
                'has_long_number': int(bool(re.search(r'\b\d{4,}\b', message))),
                'message_length': min(len(message), 500),
                'component_length': len(parts[5]),
                'node_hash': stable_bucket(parts[4]),
                'label': label,
            })
    frame = pd.DataFrame(rows)
    y = frame.pop('label').to_numpy(dtype=int)
    return frame.to_numpy(dtype=float), y, list(frame.columns)


def load_hdfs(log_path, label_path):
    line_pattern = re.compile(r'^(\d{6})\s+(\d{6})\s+\d+\s+(\w+)\s+([^:]+):\s+(.*)')
    block_pattern = re.compile(r'(blk_-?\d+)')
    severity = {'DEBUG': 0, 'INFO': 1, 'WARN': 2, 'WARNING': 2, 'ERROR': 3, 'FATAL': 4}
    blocks = defaultdict(lambda: {'n': 0, 'error': 0, 'warn': 0, 'components': set(), 'max': 0, 'sum': 0})
    with open(log_path, 'r', errors='replace') as stream:
        for line in stream:
            match = line_pattern.match(line)
            if not match:
                continue
            _, _, level, component, message = match.groups()
            sev = severity.get(level.upper(), 1)
            for block in block_pattern.findall(message):
                row = blocks[block]
                row['n'] += 1
                row['components'].add(component.strip())
                row['max'] = max(row['max'], sev)
                row['sum'] += sev
                row['error'] += int(sev >= 3)
                row['warn'] += int(sev == 2)
    labels = pd.read_csv(label_path)
    label_map = dict(zip(
        labels['BlockId'].astype(str),
        (labels['Label'].str.upper() == 'ANOMALY').astype(int),
    ))
    rows = []
    for block, row in blocks.items():
        if block not in label_map:
            continue
        count = max(row['n'], 1)
        rows.append({
            'n_events': row['n'],
            'n_error': row['error'],
            'n_warn': row['warn'],
            'n_components': len(row['components']),
            'max_severity': row['max'],
            'mean_severity': row['sum'] / count,
            'error_ratio': row['error'] / count,
            'warn_ratio': row['warn'] / count,
            'label': label_map[block],
        })
    frame = pd.DataFrame(rows)
    y = frame.pop('label').to_numpy(dtype=int)
    return frame.to_numpy(dtype=float), y, list(frame.columns)


def load_unsw(path):
    frame = pd.read_csv(path, low_memory=False)
    frame.columns = [column.lower().strip() for column in frame.columns]
    label_column = next((c for c in ['label', 'is_attack'] if c in frame.columns), frame.columns[-1])
    y = (pd.to_numeric(frame[label_column], errors='coerce').fillna(0) != 0).astype(int).to_numpy()
    frame.drop(columns=[label_column, 'srcip', 'dstip', 'attack_cat'], errors='ignore', inplace=True)
    for column in ['proto', 'state', 'service']:
        if column in frame.columns:
            frame[column] = LabelEncoder().fit_transform(frame[column].fillna('unknown').astype(str))
    frame = frame.apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)
    return frame.to_numpy(dtype=float), y, list(frame.columns)


def load_cicids(path):
    frame = pd.read_csv(path, low_memory=False)
    label_column = next(column for column in frame.columns if 'label' in column.lower())
    y = (frame[label_column].astype(str).str.strip().str.upper() != 'BENIGN').astype(int).to_numpy()
    frame.drop(columns=[label_column], inplace=True)
    frame.drop(columns=[
        column for column in frame.columns
        if any(key in column.lower() for key in ['timestamp', 'flow id', 'src ip', 'dst ip'])
    ], errors='ignore', inplace=True)
    frame.columns = [column.strip().lower().replace(' ', '_').replace('/', '_per_') for column in frame.columns]
    frame = frame.apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)
    frame['__label__'] = y
    features = [column for column in frame.columns if column != '__label__']
    frame = frame.drop_duplicates(subset=features, keep='first').reset_index(drop=True)
    y = frame.pop('__label__').to_numpy(dtype=int)
    return frame.to_numpy(dtype=float), y, list(frame.columns)


def load_requested_datasets():
    requested = list(PRIMARY_DATASETS)
    if INCLUDE_CICIDS_CONTRAST:
        requested.append('CICIDS2018')
    loaded = {}
    for name in requested:
        if name == 'BGL':
            required = [PATHS['BGL']]
            loader = lambda: load_bgl(PATHS['BGL'])
        elif name == 'HDFS':
            required = [PATHS['HDFS'], PATHS['HDFS_LABEL']]
            loader = lambda: load_hdfs(PATHS['HDFS'], PATHS['HDFS_LABEL'])
        elif name == 'UNSW-NB15':
            required = [PATHS['UNSW-NB15']]
            loader = lambda: load_unsw(PATHS['UNSW-NB15'])
        else:
            required = [PATHS['CICIDS2018']]
            loader = lambda: load_cicids(PATHS['CICIDS2018'])
        missing = [str(path) for path in required if not path.exists()]
        if missing:
            raise FileNotFoundError(f'{name}: missing {missing}')
        X, y, names = loader()
        cap = stratified_cap_indices(y, MAX_SAMPLES_PER_DATASET, seed=42)
        loaded[name] = (X[cap], y[cap], names)
        print(f'{name}: {len(cap):,} rows, {X.shape[1]} features, anomaly={y[cap].mean():.3%}')
    return loaded


## Jalankan audit tanpa mengubah sumber

File ringkasan berisi jumlah dan hash indeks, tanpa baris mentah atau log. Jika kecocokan manifest gagal, hentikan analisis dan periksa versi berkas/loader.


In [ ]:
# Angka manifest Paket A yang sudah ada; dipakai hanya sebagai audit reproduksi.
EXPECTED = {
    'BGL': {'features': 11, 'positive_count': 3607},
    'HDFS': {'features': 8, 'positive_count': 878},
    'UNSW-NB15': {'features': 48, 'positive_count': 952},
}

summary = {
    'status': 'AUDIT_ONLY_NOT_AN_EXPERIMENT',
    'source_notebook': 'ER_CyRIS_Algorithmic_Diversity_Benchmark_v1.ipynb',
    'loader_source_sha256': hashlib.sha256(
        ('"""Core utilities for the ER-CyRIS algorithmic-diversity benchmark."""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\n\nimport numpy as np\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import f1_score\nfrom sklearn.model_selection import train_test_split\n\n\ndef _as_binary_vector(y) -> np.ndarray:\n    values = np.asarray(y, dtype=int).reshape(-1)\n    if len(values) < 8:\n        raise ValueError("At least eight observations are required")\n    if set(np.unique(values)) != {0, 1}:\n        raise ValueError("Binary labels must contain both 0 and 1")\n    return values\n\n\ndef make_four_way_split(y, seed: int = 42) -> dict[str, np.ndarray]:\n    """Return disjoint 55/15/15/15 stratified index partitions."""\n    labels = _as_binary_vector(y)\n    indices = np.arange(len(labels))\n    train_idx, remainder_idx = train_test_split(\n        indices,\n        train_size=0.55,\n        random_state=seed,\n        stratify=labels,\n    )\n    middle_idx, holdout_idx = train_test_split(\n        remainder_idx,\n        test_size=1.0 / 3.0,\n        random_state=seed + 1,\n        stratify=labels[remainder_idx],\n    )\n    calibration_idx, development_idx = train_test_split(\n        middle_idx,\n        test_size=0.5,\n        random_state=seed + 2,\n        stratify=labels[middle_idx],\n    )\n    return {\n        "train": np.sort(train_idx),\n        "calibration": np.sort(calibration_idx),\n        "development": np.sort(development_idx),\n        "holdout": np.sort(holdout_idx),\n    }\n\n\ndef add_gaussian_noise(X, sigma: float, seed: int) -> np.ndarray:\n    values = np.asarray(X, dtype=float)\n    if sigma < 0:\n        raise ValueError("sigma must be non-negative")\n    if sigma == 0:\n        return values.copy()\n    noise = np.random.default_rng(seed).normal(0.0, sigma, size=values.shape)\n    return np.clip(values + noise, 0.0, 1.0)\n\n\ndef expected_calibration_error(y_true, probabilities, n_bins: int = 10) -> float:\n    labels = np.asarray(y_true, dtype=int).reshape(-1)\n    probs = np.clip(np.asarray(probabilities, dtype=float).reshape(-1), 0.0, 1.0)\n    if len(labels) != len(probs) or len(labels) == 0:\n        raise ValueError("Labels and probabilities must have the same non-zero length")\n    if n_bins < 1:\n        raise ValueError("n_bins must be positive")\n\n    edges = np.linspace(0.0, 1.0, n_bins + 1)\n    bin_ids = np.minimum(np.digitize(probs, edges[1:-1], right=False), n_bins - 1)\n    ece = 0.0\n    for bin_id in range(n_bins):\n        mask = bin_ids == bin_id\n        if not np.any(mask):\n            continue\n        accuracy = float(labels[mask].mean())\n        confidence = float(probs[mask].mean())\n        ece += float(mask.mean()) * abs(accuracy - confidence)\n    return float(ece)\n\n\ndef select_f1_threshold(y_true, probabilities) -> tuple[float, float]:\n    labels = np.asarray(y_true, dtype=int).reshape(-1)\n    probs = np.clip(np.asarray(probabilities, dtype=float).reshape(-1), 0.0, 1.0)\n    if len(labels) != len(probs) or len(labels) == 0:\n        raise ValueError("Labels and probabilities must have the same non-zero length")\n\n    candidates = np.unique(probs)\n    scored = [\n        (float(f1_score(labels, probs >= threshold, zero_division=0)), float(threshold))\n        for threshold in candidates\n    ]\n    best_score = max(score for score, _ in scored)\n    tied = [threshold for score, threshold in scored if score == best_score]\n    best_threshold = min(tied, key=lambda value: (abs(value - 0.5), value))\n    return float(best_threshold), float(best_score)\n\n\nclass PlattCalibrator:\n    """One-dimensional sigmoid calibration fitted only on the calibration split."""\n\n    def __init__(self, seed: int = 42):\n        self.seed = int(seed)\n        self.model = LogisticRegression(random_state=self.seed, solver="lbfgs")\n        self._fitted = False\n\n    def fit(self, scores, y):\n        score_array = np.asarray(scores, dtype=float).reshape(-1, 1)\n        labels = np.asarray(y, dtype=int).reshape(-1)\n        if len(score_array) != len(labels) or set(np.unique(labels)) != {0, 1}:\n            raise ValueError("Calibration data must contain aligned binary classes")\n        self.model.fit(score_array, labels)\n        self._fitted = True\n        return self\n\n    def predict_proba(self, scores) -> np.ndarray:\n        if not self._fitted:\n            raise RuntimeError("Call fit before predict_proba")\n        score_array = np.asarray(scores, dtype=float).reshape(-1, 1)\n        return self.model.predict_proba(score_array)[:, 1]\n\n\ndef stratified_cap_indices(y, max_samples: int, seed: int) -> np.ndarray:\n    labels = _as_binary_vector(y)\n    if max_samples < 2:\n        raise ValueError("max_samples must be at least two")\n    if len(labels) <= max_samples:\n        return np.arange(len(labels), dtype=int)\n    chosen, _ = train_test_split(\n        np.arange(len(labels)),\n        train_size=int(max_samples),\n        random_state=seed,\n        stratify=labels,\n    )\n    return np.sort(chosen)\n\n\ndef expected_run_count(\n    n_datasets: int,\n    n_models: int,\n    n_representations: int,\n    n_seeds: int,\n    n_conditions: int,\n) -> dict[str, int]:\n    model_fits = int(n_datasets * n_models * n_representations * n_seeds)\n    return {\n        "model_fits": model_fits,\n        "evaluation_rows": int(model_fits * n_conditions),\n    }\n\n\ndef build_run_id(\n    dataset: str,\n    representation: str,\n    model: str,\n    seed: int,\n    condition: str,\n    split: str,\n) -> str:\n    return (\n        f"{dataset}__{representation}__{model}__seed-{int(seed)}"\n        f"__{condition}__{split}"\n    )\n\n\ndef freeze_registry_digest(records: list[dict]) -> str:\n    canonical_records = sorted(\n        (json.dumps(record, sort_keys=True, separators=(",", ":")) for record in records)\n    )\n    payload = "\\n".join(canonical_records).encode("utf-8")\n    return hashlib.sha256(payload).hexdigest()\ndef stable_bucket(value, modulus=1000):\n    digest = hashlib.blake2b(str(value).encode(\'utf-8\'), digest_size=8).digest()\n    return int.from_bytes(digest, \'big\') % modulus\n\n\ndef load_bgl(path):\n    severity = {\n        \'INFO\': 1, \'DEBUG\': 0, \'WARN\': 2, \'WARNING\': 2, \'ERROR\': 3,\n        \'SEVERE\': 4, \'FATAL\': 4, \'APPFPO\': 3, \'KERN\': 2,\n        \'KERNSVC\': 3, \'APPFPW\': 2,\n    }\n    attack_words = [\n        \'fatal\', \'error\', \'fail\', \'exception\', \'abort\', \'illegal\',\n        \'segfault\', \'panic\', \'crash\', \'out of memory\', \'overflow\',\n    ]\n    rows = []\n    with open(path, \'r\', errors=\'replace\') as stream:\n        for line_number, line in enumerate(stream):\n            if BGL_PARSE_LIMIT and line_number >= BGL_PARSE_LIMIT:\n                break\n            parts = line.split(None, 8)\n            if len(parts) < 9:\n                continue\n            label = 0 if parts[0] == \'-\' else 1\n            sev = severity.get(parts[6].upper(), 1)\n            message = parts[8].lower()\n            try:\n                hour = int(parts[3].split(\':\')[0]) if \':\' in parts[3] else 12\n            except Exception:\n                hour = 12\n            rows.append({\n                \'severity\': sev,\n                \'is_error\': int(sev >= 3),\n                \'is_warning\': int(sev == 2),\n                \'hour\': hour,\n                \'off_hour\': int(hour < 6),\n                \'peak_hour\': int(8 <= hour <= 18),\n                \'attack_keyword\': int(any(word in message for word in attack_words)),\n                \'has_long_number\': int(bool(re.search(r\'\\b\\d{4,}\\b\', message))),\n                \'message_length\': min(len(message), 500),\n                \'component_length\': len(parts[5]),\n                \'node_hash\': stable_bucket(parts[4]),\n                \'label\': label,\n            })\n    frame = pd.DataFrame(rows)\n    y = frame.pop(\'label\').to_numpy(dtype=int)\n    return frame.to_numpy(dtype=float), y, list(frame.columns)\n\n\ndef load_hdfs(log_path, label_path):\n    line_pattern = re.compile(r\'^(\\d{6})\\s+(\\d{6})\\s+\\d+\\s+(\\w+)\\s+([^:]+):\\s+(.*)\')\n    block_pattern = re.compile(r\'(blk_-?\\d+)\')\n    severity = {\'DEBUG\': 0, \'INFO\': 1, \'WARN\': 2, \'WARNING\': 2, \'ERROR\': 3, \'FATAL\': 4}\n    blocks = defaultdict(lambda: {\'n\': 0, \'error\': 0, \'warn\': 0, \'components\': set(), \'max\': 0, \'sum\': 0})\n    with open(log_path, \'r\', errors=\'replace\') as stream:\n        for line in stream:\n            match = line_pattern.match(line)\n            if not match:\n                continue\n            _, _, level, component, message = match.groups()\n            sev = severity.get(level.upper(), 1)\n            for block in block_pattern.findall(message):\n                row = blocks[block]\n                row[\'n\'] += 1\n                row[\'components\'].add(component.strip())\n                row[\'max\'] = max(row[\'max\'], sev)\n                row[\'sum\'] += sev\n                row[\'error\'] += int(sev >= 3)\n                row[\'warn\'] += int(sev == 2)\n    labels = pd.read_csv(label_path)\n    label_map = dict(zip(\n        labels[\'BlockId\'].astype(str),\n        (labels[\'Label\'].str.upper() == \'ANOMALY\').astype(int),\n    ))\n    rows = []\n    for block, row in blocks.items():\n        if block not in label_map:\n            continue\n        count = max(row[\'n\'], 1)\n        rows.append({\n            \'n_events\': row[\'n\'],\n            \'n_error\': row[\'error\'],\n            \'n_warn\': row[\'warn\'],\n            \'n_components\': len(row[\'components\']),\n            \'max_severity\': row[\'max\'],\n            \'mean_severity\': row[\'sum\'] / count,\n            \'error_ratio\': row[\'error\'] / count,\n            \'warn_ratio\': row[\'warn\'] / count,\n            \'label\': label_map[block],\n        })\n    frame = pd.DataFrame(rows)\n    y = frame.pop(\'label\').to_numpy(dtype=int)\n    return frame.to_numpy(dtype=float), y, list(frame.columns)\n\n\ndef load_unsw(path):\n    frame = pd.read_csv(path, low_memory=False)\n    frame.columns = [column.lower().strip() for column in frame.columns]\n    label_column = next((c for c in [\'label\', \'is_attack\'] if c in frame.columns), frame.columns[-1])\n    y = (pd.to_numeric(frame[label_column], errors=\'coerce\').fillna(0) != 0).astype(int).to_numpy()\n    frame.drop(columns=[label_column, \'srcip\', \'dstip\', \'attack_cat\'], errors=\'ignore\', inplace=True)\n    for column in [\'proto\', \'state\', \'service\']:\n        if column in frame.columns:\n            frame[column] = LabelEncoder().fit_transform(frame[column].fillna(\'unknown\').astype(str))\n    frame = frame.apply(pd.to_numeric, errors=\'coerce\').replace([np.inf, -np.inf], np.nan)\n    return frame.to_numpy(dtype=float), y, list(frame.columns)\n\n\ndef load_cicids(path):\n    frame = pd.read_csv(path, low_memory=False)\n    label_column = next(column for column in frame.columns if \'label\' in column.lower())\n    y = (frame[label_column].astype(str).str.strip().str.upper() != \'BENIGN\').astype(int).to_numpy()\n    frame.drop(columns=[label_column], inplace=True)\n    frame.drop(columns=[\n        column for column in frame.columns\n        if any(key in column.lower() for key in [\'timestamp\', \'flow id\', \'src ip\', \'dst ip\'])\n    ], errors=\'ignore\', inplace=True)\n    frame.columns = [column.strip().lower().replace(\' \', \'_\').replace(\'/\', \'_per_\') for column in frame.columns]\n    frame = frame.apply(pd.to_numeric, errors=\'coerce\').replace([np.inf, -np.inf], np.nan)\n    frame[\'__label__\'] = y\n    features = [column for column in frame.columns if column != \'__label__\']\n    frame = frame.drop_duplicates(subset=features, keep=\'first\').reset_index(drop=True)\n    y = frame.pop(\'__label__\').to_numpy(dtype=int)\n    return frame.to_numpy(dtype=float), y, list(frame.columns)\n\n\ndef load_requested_datasets():\n    requested = list(PRIMARY_DATASETS)\n    if INCLUDE_CICIDS_CONTRAST:\n        requested.append(\'CICIDS2018\')\n    loaded = {}\n    for name in requested:\n        if name == \'BGL\':\n            required = [PATHS[\'BGL\']]\n            loader = lambda: load_bgl(PATHS[\'BGL\'])\n        elif name == \'HDFS\':\n            required = [PATHS[\'HDFS\'], PATHS[\'HDFS_LABEL\']]\n            loader = lambda: load_hdfs(PATHS[\'HDFS\'], PATHS[\'HDFS_LABEL\'])\n        elif name == \'UNSW-NB15\':\n            required = [PATHS[\'UNSW-NB15\']]\n            loader = lambda: load_unsw(PATHS[\'UNSW-NB15\'])\n        else:\n            required = [PATHS[\'CICIDS2018\']]\n            loader = lambda: load_cicids(PATHS[\'CICIDS2018\'])\n        missing = [str(path) for path in required if not path.exists()]\n        if missing:\n            raise FileNotFoundError(f\'{name}: missing {missing}\')\n        X, y, names = loader()\n        cap = stratified_cap_indices(y, MAX_SAMPLES_PER_DATASET, seed=42)\n        loaded[name] = (X[cap], y[cap], names)\n        print(f\'{name}: {len(cap):,} rows, {X.shape[1]} features, anomaly={y[cap].mean():.3%}\')\n    return loaded\n').encode('utf-8')
    ).hexdigest(),
    'dataset_manifest': {},
}

for name in PRIMARY_DATASETS:
    print('Memeriksa', name, '— membaca sumber Drive tanpa menyalinnya ke luar Colab')
    if name == 'BGL':
        X, y, feature_names = load_bgl(PATHS['BGL'])
        related = ['BGL']
    elif name == 'HDFS':
        X, y, feature_names = load_hdfs(PATHS['HDFS'], PATHS['HDFS_LABEL'])
        related = ['HDFS', 'HDFS_LABEL']
    else:
        X, y, feature_names = load_unsw(PATHS['UNSW-NB15'])
        related = ['UNSW-NB15']

    cap = stratified_cap_indices(y, MAX_SAMPLES_PER_DATASET, seed=42)
    outside = np.ones(len(y), dtype=bool)
    outside[cap] = False
    missing = np.mean(~np.isfinite(X), axis=1)
    source_files = {k: {'name': PATHS[k].name, 'bytes': PATHS[k].stat().st_size} for k in related}
    row = {
        'sources': source_files,
        'eligible_rows_in_loader_scope': int(len(y)),
        'features': int(X.shape[1]),
        'sample_cap_rows': int(len(cap)),
        'sample_cap_positives': int(np.count_nonzero(y[cap])),
        'unsampled_loader_rows': int(np.count_nonzero(outside)),
        'unsampled_loader_positives': int(np.count_nonzero(y[outside])),
        'sample_cap_index_sha256': hashlib.sha256(cap.astype('int64').tobytes()).hexdigest(),
        'sample_cap_missing_fraction_min': float(missing[cap].min()),
        'sample_cap_missing_fraction_max': float(missing[cap].max()),
        'sample_cap_rows_with_missing': int(np.count_nonzero(missing[cap] > 0)),
        'unsampled_rows_with_missing': int(np.count_nonzero(missing[outside] > 0)),
        'source_matches_previous_cap_manifest': bool(
            len(cap) == 30_000
            and X.shape[1] == EXPECTED[name]['features']
            and np.count_nonzero(y[cap]) == EXPECTED[name]['positive_count']
        ),
    }
    summary['dataset_manifest'][name] = row
    print(json.dumps({name: row}, indent=2, ensure_ascii=False))
    del X, y, missing, cap, outside

summary['all_sample_caps_match'] = all(
    row['source_matches_previous_cap_manifest']
    for row in summary['dataset_manifest'].values()
)
summary['has_unsampled_eligible_rows'] = all(
    row['unsampled_loader_rows'] > 0
    for row in summary['dataset_manifest'].values()
)
summary['provenance_status'] = (
    'Sumber diketahui per dataset, tetapi kualitas sumber dan parsing untuk '
    'semua alert belum terukur. Variasi missingness tidak otomatis menjadi q_e.'
)
summary['evaluation_status'] = (
    'Baris di luar cap hanya kandidat evaluasi. Belum diaudit duplikasi, '
    'pemisahan waktu atau kelompok, kualitas per alert, atau keterkaitan label.'
)

path = OUT_DIR / 'preflight_summary.json'
path.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
print('HASIL:', path)
print('Kirim hanya preflight_summary.json; jangan kirim log mentah.')
if not summary['all_sample_caps_match']:
    print('STOP: file/loader tidak mereproduksi manifest 30.000 sampel lama.')
elif not summary['has_unsampled_eligible_rows']:
    print('STOP: tidak ada kandidat sisa di cakupan loader yang dipakai.')
else:
    print('LANJUT AUDIT: kandidat sisa ada; belum boleh disebut holdout konfirmatori.')
